In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np
from scipy.stats import f_oneway, chi2_contingency
import pickle
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler,MinMaxScaler
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances_argmin_min
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.patches as mpatches

#### Read in Dataset

In [ ]:
with open(r'med_lab_static_output_12h_imputed.pkl', 'rb') as handle:
    med_lab_static_output = pickle.load(handle)

#### Trajectory

In [ ]:
def add_trajectory_features(df):
    """
    Add trajectory features for BG_VALUE within each CSN, including changes, slope, and variability.
    """
    # Remove rows where BG_VALUE > 1000
    df = df[df['BG_VALUE'] <= 1000]
    df = df.sort_values(by=['CSN', 'BG_RESULT_TIME'])
    # Calculate time difference in seconds
    df['TIME_DIFF'] = df.groupby('CSN')['BG_RESULT_TIME'].diff().dt.total_seconds()
    # Calculate BG change
    df['BG_CHANGE'] = df.groupby('CSN')['BG_VALUE'].diff()
    # Calculate BG slope (change in BG per unit time)
    df['BG_SLOPE'] = df['BG_CHANGE'] / df['TIME_DIFF']
    # Calculate BG variability (std deviation of BG within each CSN)
    df['BG_VARIABILITY'] = df.groupby('CSN')['BG_VALUE'].transform(lambda x: x.std())
    # Replace NaN values resulting from diff() with 0 (e.g., first row of each group)
    df.fillna(0, inplace=True)
    return df
# Apply to your dataset
df = add_trajectory_features(med_lab_static_output)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print('df shape before drop nan', df.shape)
df = df.dropna()
print('df shape after drop nan', df.shape)

#### Propensity Score Matching 

In [ ]:
results = []  # To store results for each drug

# Filter numeric columns for processing
drug_columns = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]

# Loop through each column in the filtered list
for drug in sorted(drug_columns):
    # Skip non-drug columns
    if drug in ['AGE','BG_CHANGE','BG_SLOPE','CSN','TIME_DIFF',
                'BG_VALUE','BG_VARIABILITY', 'SEX_Male', 'PMH_CHF','T1DM','T2DM', "Admit_Pain", "Admit_Sepsis", "PMH_CKD"]:
        continue

    # Aggregate data
    agg_data = df.groupby('CSN').agg({
        'SEX_Male': 'max',
        'AGE': 'mean',
        'PMH_CHF': 'max',
        drug: 'mean',  # Use the current drug
        'BG_VALUE': 'mean',
        'PMH_T1DM': 'max',
        'PMH_T2DM': 'max',
        'BG_CHANGE': 'mean',
        'BG_VARIABILITY': 'mean',
        'PMH_CKD':'max',
        'Admit_Pain':'max',
        'Admit_Sepsis':'max'
    }).reset_index()

    # Create a binary treatment indicator for the current drug
    agg_data['treated'] = (agg_data[drug] > 0).astype(int)
    covariates = ['AGE', 'SEX_Male', 'PMH_T1DM', 'PMH_T2DM', 'PMH_CHF', 'PMH_CKD', 'Admit_Pain', 'Admit_Sepsis']
    logistic = LogisticRegression(max_iter=1000, random_state=42)
    # Check the number of unique classes in the 'treated' column
    if agg_data['treated'].nunique() < 2:
        print("Skipping logistic regression as there is only one class in the 'treated' column.")
        continue
     # Fit the logistic regression model
    agg_data['propensity_score'] = logistic.fit(
        agg_data[covariates], agg_data['treated']).predict_proba(agg_data[covariates])[:, 1]
    
    # Match treated and control patients
    treated = agg_data[agg_data['treated'] == 1]
    control = agg_data[agg_data['treated'] == 0]

    treated_ps = treated['propensity_score'].values.reshape(-1, 1)
    control_ps = control['propensity_score'].values.reshape(-1, 1)

    control_matches = pairwise_distances_argmin_min(treated_ps, control_ps, metric='euclidean')[0]
    matched_control_indices = control.index[control_matches]
    matched_indices = treated.index.union(matched_control_indices)
    matched_data = agg_data.loc[matched_indices]

    # Ensure binary variables are properly typed
    matched_data['SEX_Male'] = matched_data['SEX_Male'].astype(int)

# Build regression model
    X = sm.add_constant(matched_data[['treated', 'AGE', 'SEX_Male', 'PMH_CHF', 'PMH_T1DM', 'PMH_T2DM','PMH_CKD','Admit_Pain',
                                     'Admit_Sepsis']])
    y = matched_data['BG_CHANGE']
            
    # Fit the regression model
    model = sm.OLS(y, X).fit()
    
    # Record the coefficient, p-value for 'treated', and the overall R^2
    coef = model.params['treated']
    p_value = model.pvalues['treated']
    r2 = model.rsquared  # Add R^2
    results.append({'drug': drug, 'coef': coef, 'p_value': p_value, 'r2': r2})

ps_results_df = pd.DataFrame(results)
# Apply Benjamini-Hochberg correction to p-values
p_values = ps_results_df['p_value']
corrected = multipletests(p_values, method='fdr_bh')
ps_results_df['adjusted_p_value'] = corrected[1]  # Add adjusted p-values to DataFrame

In [ ]:
ps_sig01=ps_results_df[ps_results_df['adjusted_p_value']<0.01]

In [ ]:
# Ensure p-values are not zero (to avoid issues with log scale)
ps_results_df['adjusted_p_value'] = ps_results_df['adjusted_p_value'].replace(0, 1e-10)

# Compute -log10 of adjusted p-values for the y-axis
ps_results_df['log_p'] = -np.log10(ps_results_df['adjusted_p_value'])

# Define significance threshold (p < 0.01)
threshold = -np.log10(0.01)

# Create figure
#plt.figure(figsize=(10, 6))

# Scatter plot for all drugs
plt.scatter(ps_results_df['coef'], ps_results_df['log_p'], color='gray', alpha=0.7, label="Non-significant")

# Highlight significant medications (adjusted p < 0.01)
sig_meds = ps_results_df[ps_results_df['adjusted_p_value'] < 0.01]
plt.scatter(sig_meds['coef'], sig_meds['log_p'], color='red', alpha=0.8, label="Significant (p < 0.01)")

# Add threshold line
plt.axhline(threshold, color='black', linestyle='--', linewidth=1, label="p = 0.01")

# Labels and formatting
plt.xlabel("Coefficient", fontsize=25)
plt.ylabel("-Log10 Adjusted P-value", fontsize=20)
#plt.title("Volcano Plot of Medication Effects on BG Change", fontsize=16)
plt.legend(fontsize=15)
plt.xticks(fontsize=18)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=18) 
plt.tight_layout()
plt.savefig(r'figures_revision1\vol_plot.svg')
plt.show()

In [ ]:
# Ensure p-values are not zero (to avoid log scale issues)
ps_results_df['adjusted_p_value'] = ps_results_df['adjusted_p_value'].replace(0, 1e-10)

# Compute -log10 of adjusted p-values for the y-axis
ps_results_df['log_p'] = -np.log10(ps_results_df['adjusted_p_value'])

# Define significance threshold (p < 0.01)
threshold = -np.log10(0.01)

# Identify points where y > 20 and (x > 2 or x < -2)
label_points = ps_results_df[(ps_results_df['log_p'] > 10)]

# Create figure
plt.figure(figsize=(10, 6))

# Scatter plot for all drugs
plt.scatter(ps_results_df['coef'], ps_results_df['log_p'], color='gray', alpha=0.7, label="Non-significant")

# Highlight significant medications (adjusted p < 0.01)
sig_meds = ps_results_df[ps_results_df['adjusted_p_value'] < 0.01]
plt.scatter(sig_meds['coef'], sig_meds['log_p'], color='red', alpha=0.8, label="Significant (p < 0.01)")

# Add threshold line
plt.axhline(threshold, color='black', linestyle='--', linewidth=1, label="p = 0.01")

# Label the selected points with 'Feature' column values
for i, row in label_points.iterrows():
    plt.text(row['coef'], row['log_p'], row['Feature'], fontsize=12, color='darkgreen', ha='right')

# Labels and formatting
plt.xlabel("Coefficient", fontsize=25)
plt.ylabel("-Log10 Adjusted P-value", fontsize=20)
plt.legend(fontsize=15)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()

# Show the updated plot
plt.show()

In [ ]:
PSM_table=pd.read_excel(r'PSM_sig.xlsx')

In [ ]:
# Define a function to check if the known BG direction matches the coefficient sign
def check_direction_match(row):
    """Returns green if the known direction matches the coefficient sign, red if mismatched, and black if NaN."""
    if pd.isna(row["Known BG Direction"]):  # If NaN, return black
        return "black"
    elif (row["Known BG Direction"].strip().lower() == "up" and row["Coefficient"] > 0) or \
         (row["Known BG Direction"].strip().lower() == "down" and row["Coefficient"] < 0):
        return "green"  
    else:
        return "red"  

# Apply the function to determine text color for each drug
PSM_table["Text Color"] = PSM_table.apply(check_direction_match, axis=1)

In [ ]:
# Reshape the data using pivot_table
heatmap_data = PSM_table.pivot_table(index="drug", columns="Known BG Effect", values="Coefficient")

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(
    heatmap_data, 
    cmap="coolwarm", 
    annot=True, 
    fmt=".2f", 
    linewidths=0.5, 
    annot_kws={"size": 10},
    ax=ax
)

# Ensure y-ticks align correctly with the heatmap grid
ax.set_yticks(np.arange(len(heatmap_data.index)) + 0.5)
ax.set_yticklabels(heatmap_data.index, fontsize=12, rotation=0)  # Keep labels in correct place

# Rotate x-labels for readability
ax.set_xticklabels(ax.get_xticklabels(), fontsize=12,  ha="right")

# Label the colorbar as "Coefficient"
cbar = ax.collections[0].colorbar
cbar.set_label("Coefficient", fontsize=14)

# Apply color directly to y-axis tick labels (without moving them)
for label in ax.get_yticklabels():
    drug_name = label.get_text()
    text_color = PSM_table.loc[PSM_table["drug"] == drug_name, "Text Color"].values[0]  # Get correct color
    label.set_color(text_color)  # Apply color directly to the tick label

# Create a custom legend for Known BG Effect Agreement
legend_patches = [
    mpatches.Patch(color="green", label="Agrees"),
    mpatches.Patch(color="red", label="Disagrees")
]

# Move the legend **outside** the plot (right side)
fig.legend(
    handles=legend_patches, 
    title="BG Direction", 
    loc="upper right", 
    bbox_to_anchor=(0.2, 1),  # Move legend outside
    fontsize=12, 
    title_fontsize=14
)

# Formatting
ax.set_xlabel("Prior BG Effect Documented?", fontsize=14)
ax.set_ylabel("Drug/Clinical Variable", fontsize=14)
#ax.set_title("Drug and Clinical Variable Effects on BG Levels", fontsize=16)

# Adjust layout to fit all labels properly and allow space for the legend
plt.tight_layout()  # Leaves space for legend
plt.savefig(r'figures_revision1\heatmap.svg')
# Show the heatmap
plt.show()

In [ ]:
ps_sig01.to_excel('PSM_sig.xlsx')

In [ ]:
plt.figure(figsize=(12, 4))


# Plot the barplot using seaborn
sns.barplot(data=ps_sig01, x='drug', y='coef', color='blue')

# Rotate x-axis labels
plt.xticks(rotation=90, fontsize=12)  # Adjust the angle and font size for x-ticks
plt.yticks(fontsize=14)  # Adjust font size for y-ticks

# Adjust layout for better appearance
plt.tight_layout()

# Save the plot to a file
#plt.savefig('figures_revision1/lasso_12h.svg')

# Show the plot
plt.show()

In [ ]:
ps_results_df.rename(columns={"drug": "Feature"}, inplace=True)

In [ ]:
ps_results_df.to_csv('psm_results_df.csv')

In [ ]:
class_effect_df=pd.read_excel(r'avg_coefficients_lasso_1h.xlsx')

In [ ]:
ps_class_effect=pd.merge(class_effect_df,ps_results_df, on='Feature') 

In [ ]:
ps_class_effect['abs coef'] = ps_class_effect['coef'].abs()

In [ ]:
# Count the total rows where average coefficient is exactly 0
total_zero_coeff = ps_class_effect[ps_class_effect["abs coef"] == 0].shape[0]

# Count how many of those rows have BG effect == "yes"
yes_count = ps_class_effect[(ps_class_effect["abs coef"] == 0) & (ps_class_effect["BG effect"] == "yes")].shape[0]

# Calculate the percentage
percentage = (yes_count / total_zero_coeff) * 100 if total_zero_coeff > 0 else 0

print(f"Percentage of rows with average coefficient = 0 that had a BG effect of 'yes': {percentage:.2f}%")

In [ ]:
cutoffs = np.concatenate([[0], np.linspace(0, 1, 20), np.linspace(1.2, ps_class_effect['abs coef'].max(), 10)])
percent_yes = [0] + [
    (ps_class_effect[ps_class_effect['abs coef'] >= cutoff]['BG effect'] == 'yes').mean() * 100
    for cutoff in cutoffs[1:]
]

# Create the plot
plt.figure(figsize=(4.5, 3))
plt.step(cutoffs, percent_yes, where='post', color='b', linewidth=2, label="% Known BG Effect")

# Formatting
plt.xlabel("Absolute Coefficient Threshold", fontsize=14)
plt.ylabel("% Features with Known Effect", fontsize=11)
plt.xticks(np.arange(0, 2.5, 0.2),fontsize=11)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=11) 
plt.xlim(-0.1, 2.3)  # Adjust x-axis limits to include the full range cleanly
plt.ylim(0, 105)  # Keep percentage within reasonable bounds
#plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

plt.savefig(r'figures_revision1\psm_effect.svg')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.barplot(data=ps_class_effect.sort_values(by='Class'), x='Class', y='coef', color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
#plt.xlabel('Drug Class',fontsize=15)
plt.ylabel("Coefficient",fontsize=17)
plt.xlabel("Drug Class",fontsize=17)
# Adjust layout for clarity
plt.xticks(fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=15) 
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.tight_layout()
plt.savefig(r'figures_revision1\psm_class.svg')
plt.show()

In [ ]:
keywords = ["antidiabetic"]

# Use regex to match any of the keywords (case-insensitive)
filtered_df = ps_class_effect[
    ps_class_effect['Class'].str.contains('|'.join(keywords), case=False, na=False)]

In [ ]:
filtered_df["Feature"] = ['glimepiride 1mg PO',
 'glimepiride 2mg PO',
 'glimepiride 4mg PO',
 'glipizide 5mg PO',
 'glipizide 5mg PO ER',
 'glipizide 10mg PO',
 'glipizide 10mg PO ER',
 'glyburide 1.25mg PO',
 'glyburide 5mg PO',
 'insulin aspart SQ',
 'insulin glargine SQ',
 'insulin lispro SQ',
 'insulin NPH & regular 70/30 SQ',
 'insulin NPH SQ',
 'insulin regular SQ',
 'insulin regular IV',
 'metformin 500mg PO',
 'metformin 1000mg PO',
 'nateglinide 60mg PO',
 'nateglinide 120mg PO',
 'pioglitazone 15mg PO',
 'pioglitazone 30mg PO',
 'pioglitazone 45mg PO',
 'sitagliptin 25mg PO',
 'sitagliptin 50mg PO',
 'sitagliptin 100mg PO']
                                                                  
plt.figure(figsize=(12, 5))
sns.barplot(data=filtered_df, x='Feature', y='coef', color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
#plt.xlabel('Drug Class',fontsize=15)
plt.ylabel("Coefficient",fontsize=15)
plt.xlabel("Drug",fontsize=15)
# Adjust layout for clarity
plt.tight_layout()
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.xticks(fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=15) 

#plt.legend(title="Drug Class", title_fontsize=12, fontsize=12, loc="best")  # Adjust size
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.tight_layout()
plt.savefig(r'figures_revision1\psm_dm.svg')
plt.show()